In [3]:
with open("combined_text.txt", "r", encoding="utf-8") as f:
    text_sequence = f.read()

len(text_sequence)

682464

In [4]:
import sys 
sys.path.append("..")

In [5]:
from minbpe import BasicTokenizer

tokenizer = BasicTokenizer()
tokenizer.train(text_sequence, vocab_size=1024)

100%|██████████| 768/768 [04:15<00:00,  3.01it/s]


In [6]:
vocab = tokenizer.vocab
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [9]:
tokenizer.encode("why win")

[966, 695]

In [10]:
tokenizer.encode("testing things out to see if it all works")

[116, 446, 276, 268, 276, 258, 659, 296, 628, 545, 388, 434, 559, 107, 115]

In [11]:
tokenizer.decode([116, 446, 276, 268, 276, 258, 659, 296, 628, 545, 388, 434, 559, 107, 115])

'testing things out to see if it all works'

Add special tokens to the vocabulary. These tokens are going to be used a lot in the fine-tuning step.

In [12]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5,
}

In [13]:
len(tokenizer.vocab.keys())

1024

In [ ]:

len(tokenizer.encode(text_sequence))

In [ ]:

tokenizer.save(file_prefix="../output/tokenizer/my_tokenizer")

In [1]:
import random

In [ ]:
class Matrix:
    def __init__(self, matrix):
        self.data = [list(row) for row in matrix]
        self.rows = len(matrix)
        self.cols = len(matrix[0])
        self.shape = (self.rows, self.cols)

    def __str__(self):
        return '\n'.join([' '.join([str(elem) for elem in row]) for row in self.data])

    def __repr__(self):
        col_widths = []
        for j in range(self.cols):
            width = max(len(f"{self.data[i][j]:.4f}") for i in range(self.rows))
            col_widths.append(width)
        lines = []
        for i in range(self.rows):
            row_str = "  ".join(
                f"{self.data[i][j]:{col_widths[j]}.4f}" for j in range(self.cols)
            )
            bracket_l = "|" if 0 < i < self.rows - 1 else ("/" if i == 0 else "\\")
            bracket_r = "|" if 0 < i < self.rows - 1 else ("\\" if i == 0 else "/")
            lines.append(f"  {bracket_l} {row_str} {bracket_r}")
        print(lines)
        header = f"Matrix {self.rows}x{self.cols}:"
        return header + "\n" + "\n".join(lines)

    def __add__(self, other):
        if self.shape != other.shape:
            raise ValueError("Matrices must have the same shape for addition.")
        else:
            if isinstance(other, Matrix):
                if self.shape == other.shape:
                    output = []
                    for i in range(self.rows):
                        row = []
                        for j in range(self.cols):
                            row.append(self.data[i][j] + other.data[i][j])

                        output.append(row)
                        
                    return Matrix(output)
                    
                if other.rows == 1 and other.cols == self.cols:
                    return Matrix([
                        [self.data[i][j] + other.data[0][j] for j in range(self.cols)]
                        for i in range(self.rows)
                    ])
                if other.cols == 1 and other.rows == self.rows:
                    return Matrix([
                        [self.data[i][j] + other.data[i][0] for j in range(self.cols)]
                        for i in range(self.rows)
                ])
        raise ValueError(f"Cannot add shapes {self.shape} and {other.shape}")

    def __sub__(self, other):
        if self.shape != other.shape:
            raise ValueError("Matrices must have the same shape for subtraction.")
        else:
            return Matrix([[self.data[i][j] - other.data[i][j] 
                for j in range(self.cols)] for i in range(self.rows)])
        
    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])
    
    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])
                

    def matmul(self, other):
        if self.cols != other.rows:
            raise ValueError(
                f"Cannot multiply shapes {self.shape} and {other.shape}: "
                f"inner dimensions {self.cols} != {other.rows}"
            )
        
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])
    
    def __matmul__(self, other):
        return self.matmul(other)
    
    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])
    
    @property
    def T(self):
        return self.transpose()
    
    @staticmethod
    def identity(n):
        return Matrix([
            [i if i == j else 0 for j in range(n)]
            for i in range(n)
        ])
    
    @staticmethod
    def zeros(rows, cols):
        return Matrix([[0]*cols for _ in range(rows)])

    @staticmethod
    def random(rows, cols, low=-1.0, high=1.0):
        return Matrix([
            [random.uniform(low, high) for _ in range(cols)]
            for _ in range(rows)
        ])
        

Matrix([[1, 2, 3], [4, 5, 6], [6, 7, 8]]) + Matrix([[0, 0, 1], [0, 0, 1], [0, 0, 1]])

['  / 1.0000  2.0000  4.0000 \\', '  | 4.0000  5.0000  7.0000 |', '  \\ 6.0000  7.0000  9.0000 /']


Matrix 3x3:
  / 1.0000  2.0000  4.0000 \
  | 4.0000  5.0000  7.0000 |
  \ 6.0000  7.0000  9.0000 /